# 📊 Day 4 — Statistics & EDA Project: Titanic Dataset

**Objective:** Apply statistical thinking to produce a full EDA report with 10 observations,
a rigorous hypothesis test, and a publishable GitHub Pages mini-site.

**Deliverables:**
- `titanic_profile.html` — automated ydata-profiling report
- 10 documented EDA observations with supporting visualizations
- Hypothesis test: Does passenger class significantly affect survival odds?
- `docs/index.html` — GitHub Pages mini-site

| Section | Content |
|---------|---------|
| §1 | Environment setup |
| §2 | Data loading & overview |
| §3 | ydata-profiling automated report |
| §4 | 10 EDA observations + plots |
| §5 | Hypothesis testing (t-test + chi-square + Mann-Whitney U) |
| §6 | Executive summary & export |

## §1 — Environment Check

In [ ]:
%matplotlib inline
import sys, warnings, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# ── Directories ──────────────────────────────────────────────────────────────
DATA_DIR    = Path("data")
FIG_DIR     = Path("figures")
DOCS_DIR    = Path("docs")
for d in [DATA_DIR, FIG_DIR, DOCS_DIR]:
    d.mkdir(exist_ok=True)

# ── Versions ─────────────────────────────────────────────────────────────────
print("=" * 50)
print("ENVIRONMENT")
print("=" * 50)
print(f"Python       : {sys.version.split()[0]}")
print(f"Pandas       : {pd.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"SciPy        : {stats.__version__ if hasattr(stats, '__version__') else 'ok'}")
try:
    import ydata_profiling as ydp
    print(f"ydata-prof.  : {ydp.__version__}")
except ImportError:
    print("ydata-prof.  : not installed  →  pip install ydata-profiling")

# ── Plot defaults ─────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="colorblind", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False})

SURVIVE_COLORS = {0: "#d62728", 1: "#2ca02c"}   # red=died, green=survived
print("\n✅ Environment ready")

---
## §2 — Load Raw Data

We load the raw Titanic CSV (no pre-cleaning) to perform authentic EDA — missing values
and outliers are part of the story.

In [ ]:
# ── Download if needed ───────────────────────────────────────────────────────
RAW_PATH = DATA_DIR / "titanic.csv"

if not RAW_PATH.exists():
    import urllib.request
    url = ("https://raw.githubusercontent.com/"
           "datasciencedojo/datasets/master/titanic.csv")
    print(f"Downloading: {url}")
    urllib.request.urlretrieve(url, RAW_PATH)
    print(f"✅ Saved: {RAW_PATH}")
else:
    print(f"✅ Found: {RAW_PATH}")

# ── Load & minimal rename ─────────────────────────────────────────────────────
df = pd.read_csv(RAW_PATH).rename(columns=str.lower).rename(columns={
    "passengerid": "passenger_id", "pclass": "pclass",
    "sibsp": "sibsp", "parch": "parch",
})
# Extract title (needed for Obs 10)
df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.")
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"]    = (df["family_size"] == 1).astype(int)

print(f"\nShape : {df.shape}")
print(f"Cols  : {df.columns.tolist()}")
print()
df.info()

In [ ]:
print("\nDescriptive statistics:")
display = __builtins__.__dict__.get("display", print)
display(df.describe(include="all").T)

print("\nMissing values:")
miss = df.isnull().sum()
print(miss[miss > 0].to_string())

---
## §3 — ydata-profiling Automated Report

`minimal=True` skips expensive correlation matrices — run completes in < 60 s.
Open `titanic_profile.html` in your browser after the cell finishes.

In [ ]:
try:
    from ydata_profiling import ProfileReport

    PROFILE_PATH = DATA_DIR / "titanic_profile.html"

    profile = ProfileReport(
        df,
        title="Titanic Dataset — ydata-profiling EDA",
        minimal=True,          # skip heavy correlation / interaction sections
        progress_bar=False,    # cleaner output in notebooks
        html={"style": {"full_width": True}},
    )
    profile.to_file(PROFILE_PATH)
    print(f"✅ Profile saved: {PROFILE_PATH}")
    print(f"   File size: {PROFILE_PATH.stat().st_size // 1024} KB")
    print("   → Open in browser or check the Docs panel in JupyterLab")

    # Also copy to docs/ for GitHub Pages
    import shutil
    shutil.copy(PROFILE_PATH, DOCS_DIR / "titanic_profile.html")
    print(f"   → Copied to docs/  for GitHub Pages")

except ImportError:
    print("⚠  ydata-profiling not installed.  Run: pip install ydata-profiling")
    print("   Continuing with manual EDA...")

---
## §4 — 10 EDA Observations

Each observation is supported by a computation and at least one visualization.
Fill in the **[placeholder]** values after running the cells.

### Observation 1 — Overall Survival Rate
> **"Only [38.4%] of the 891 passengers in the training set survived, confirming the
> high overall mortality of the disaster."**

In [ ]:
total     = len(df)
survived  = df["survived"].sum()
rate      = df["survived"].mean()

print(f"Total passengers : {total}")
print(f"Survived         : {survived}  ({rate:.1%})")
print(f"Did not survive  : {total - survived}  ({1-rate:.1%})")

fig, ax = plt.subplots(figsize=(5, 3.5))
vals    = [total - survived, survived]
labels  = [f"Did not survive\n({1-rate:.1%})", f"Survived\n({rate:.1%})"]
colors  = [SURVIVE_COLORS[0], SURVIVE_COLORS[1]]
ax.bar(labels, vals, color=colors, edgecolor="white", linewidth=1.2)
ax.set_ylabel("Passengers")
ax.set_title("Observation 1: Overall Survival", fontweight="bold")
for i, v in enumerate(vals):
    ax.text(i, v + 5, str(v), ha="center", fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "obs1_overall_survival.png", dpi=110)
plt.show()
plt.close()

### Observation 2 — Survival by Sex
> **"Female passengers survived at 3.9× the rate of males (74.2% vs 18.9%),
> reflecting the 'women and children first' evacuation protocol."**

In [ ]:
surv_sex = df.groupby("sex")["survived"].agg(["mean","sum","count"])
surv_sex.columns = ["rate","survived","total"]
print(surv_sex.to_string())

female_rate = surv_sex.loc["female","rate"]
male_rate   = surv_sex.loc["male","rate"]
print(f"\nFemale / Male rate ratio: {female_rate/male_rate:.1f}×")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
# Left: raw counts stacked
for i, (sex, grp) in enumerate(df.groupby("sex")):
    n_surv = grp["survived"].sum()
    n_dead = len(grp) - n_surv
    axes[0].bar(sex, n_dead,  color=SURVIVE_COLORS[0], label="Died"     if i==0 else "")
    axes[0].bar(sex, n_surv,  color=SURVIVE_COLORS[1], label="Survived" if i==0 else "",
                bottom=n_dead)
axes[0].set_title("Passenger Counts by Sex")
axes[0].set_ylabel("Passengers")
axes[0].legend()
# Right: survival rate
axes[1].bar(surv_sex.index, surv_sex["rate"], color=[SURVIVE_COLORS[1]]*2)
axes[1].axhline(df["survived"].mean(), ls="--", color="gray", label="Overall avg")
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_title("Survival Rate by Sex")
axes[1].legend()
plt.suptitle("Observation 2: Survival by Sex", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "obs2_survival_sex.png", dpi=110)
plt.show(); plt.close()

### Observation 3 — Survival by Passenger Class
> **"1st-class passengers survived at 63.0%, more than 2.6× the rate of
> 3rd-class passengers (24.2%), exposing severe class-based inequality in lifeboat access."**

In [ ]:
surv_cls = df.groupby("pclass")["survived"].agg(["mean","sum","count"])
surv_cls.columns = ["rate","survived","total"]
print(surv_cls.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
x = surv_cls.index.astype(str)
bars = ax.bar(x, surv_cls["rate"],
              color=[SURVIVE_COLORS[1] if r>0.5 else SURVIVE_COLORS[0]
                     for r in surv_cls["rate"]],
              edgecolor="white", linewidth=1.2)
ax.axhline(df["survived"].mean(), ls="--", color="gray", lw=1.5, label="Overall avg")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Survival Rate")
ax.set_title("Observation 3: Survival Rate by Class", fontweight="bold")
ax.legend()
for bar, val in zip(bars, surv_cls["rate"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f"{val:.1%}", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "obs3_survival_class.png", dpi=110)
plt.show(); plt.close()

### Observation 4 — Age Distribution & Missingness
> **"Age is missing for 19.9% of passengers.  Among those with known ages,
> the distribution is right-skewed (median 28, mean 29.7); children under 10
> had a notably higher survival rate than other age groups."**

In [ ]:
print(f"Age missing: {df['age'].isnull().sum()} ({df['age'].isnull().mean():.1%})")
print(df["age"].describe().to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# Distribution by survival
for survived, grp in df.dropna(subset=["age"]).groupby("survived"):
    label = "Survived" if survived else "Did not survive"
    axes[0].hist(grp["age"], bins=25, alpha=0.6,
                 color=SURVIVE_COLORS[survived], label=label, density=True)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Density")
axes[0].set_title("Age Distribution by Outcome")
axes[0].legend()

# Age band survival
df_known = df.dropna(subset=["age"]).copy()
df_known["age_band"] = pd.cut(df_known["age"],
                               bins=[0,10,18,30,45,60,100],
                               labels=["<10","10-18","18-30","30-45","45-60","60+"])
age_surv = df_known.groupby("age_band", observed=True)["survived"].mean()
age_surv.plot(kind="bar", color=SURVIVE_COLORS[1], ax=axes[1], rot=0)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_title("Survival Rate by Age Band")
axes[1].set_xlabel("Age Band")
plt.suptitle("Observation 4: Age Distribution", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "obs4_age_distribution.png", dpi=110)
plt.show(); plt.close()

### Observation 5 — Fare Distribution (Heavy Skew)
> **"Fares are extremely right-skewed (median £14.5, max £512); 1st-class fares
> are ~13× higher than 3rd-class, and high fare correlates strongly with survival."**

In [ ]:
print(df.groupby("pclass")["fare"].describe().to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# Boxplot by class
df.boxplot(column="fare", by="pclass", ax=axes[0],
           notch=False, patch_artist=True,
           boxprops=dict(facecolor="#4c72b0", alpha=0.6))
axes[0].set_title("Fare by Passenger Class")
axes[0].set_xlabel("Passenger Class")
axes[0].set_ylabel("Fare (£)")
# Exclude top 1% for readability
fare_p99 = df["fare"].quantile(0.99)
df_trim = df[df["fare"] <= fare_p99]
for survived, grp in df_trim.groupby("survived"):
    axes[1].hist(grp["fare"], bins=30, alpha=0.6,
                 color=SURVIVE_COLORS[survived],
                 label="Survived" if survived else "Did not")
axes[1].set_xlabel("Fare (£)  [capped at 99th pct]")
axes[1].set_ylabel("Count")
axes[1].set_title("Fare Distribution by Outcome")
axes[1].legend()
plt.suptitle("Observation 5: Fare Distribution", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "obs5_fare_distribution.png", dpi=110)
plt.show(); plt.close()

### Observation 6 — Class × Sex Interaction (Heatmap)
> **"The survival advantage of being female is strongest in 1st class (96.5%) and
> weakest in 3rd class (50.0%), suggesting class mediated access to lifeboats."**

In [ ]:
pivot = df.pivot_table(values="survived", index="sex",
                       columns="pclass", aggfunc="mean")
print("Survival rate (rows=sex, cols=pclass):")
print((pivot * 100).round(1).to_string())

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.heatmap(pivot, annot=True, fmt=".1%", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={"format": mticker.PercentFormatter(1.0)})
ax.set_title("Observation 6: Survival Rate — Sex × Class", fontweight="bold")
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Sex")
plt.tight_layout()
plt.savefig(FIG_DIR / "obs6_sex_class_heatmap.png", dpi=110)
plt.show(); plt.close()

### Observation 7 — Age × Survival (Violin Plot)
> **"Survivors have a notably younger median age (28 vs 30 for non-survivors);
> the violin shape reveals the distinctive spike of young children among survivors."**

In [ ]:
df_known = df.dropna(subset=["age"]).copy()
medians = df_known.groupby("survived")["age"].median()
print(f"Median age — Survived: {medians[1]:.1f}  |  Did not: {medians[0]:.1f}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.violinplot(data=df_known, x="survived", y="age",
               palette=[SURVIVE_COLORS[0], SURVIVE_COLORS[1]],
               inner="box", cut=0, ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Did not survive", "Survived"])
ax.set_xlabel("")
ax.set_ylabel("Age")
ax.set_title("Observation 7: Age Distribution by Outcome", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "obs7_age_violin.png", dpi=110)
plt.show(); plt.close()

### Observation 8 — Family Size vs. Survival
> **"Passengers traveling alone had only a 30.4% survival rate vs. 50.5% for those
> with 2–4 family members; very large families (5+) fared worst at 16.1%."**

In [ ]:
surv_fs = df.groupby("family_size")["survived"].agg(["mean","count"])
surv_fs.columns = ["rate","count"]
print(surv_fs.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colors = [SURVIVE_COLORS[1] if r >= 0.5 else SURVIVE_COLORS[0]
          for r in surv_fs["rate"]]
bars = ax.bar(surv_fs.index.astype(str), surv_fs["rate"],
              color=colors, edgecolor="white")
ax.axhline(df["survived"].mean(), ls="--", color="gray", lw=1.5, label="Overall avg")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
# Annotate with n
for bar, (fs, row) in zip(bars, surv_fs.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2,
            row["rate"] + 0.01, f"n={row['count']}", ha="center", fontsize=9)
ax.set_xlabel("Family Size (self + siblings/spouses + parents/children)")
ax.set_ylabel("Survival Rate")
ax.set_title("Observation 8: Family Size vs. Survival", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "obs8_family_size.png", dpi=110)
plt.show(); plt.close()

### Observation 9 — Embarkation Port
> **"Passengers embarking from Cherbourg (C) had the highest survival rate (55.4%),
> likely because it had the highest proportion of 1st-class passengers."**

In [ ]:
df_emb = df.dropna(subset=["embarked"]).copy()
surv_emb = df_emb.groupby("embarked")["survived"].agg(["mean","count"])
surv_emb.columns = ["rate","count"]
pclass_emb = df_emb.groupby("embarked")["pclass"].value_counts(normalize=True).unstack()
print("Survival rate by port:")
print(surv_emb.to_string())
print("\nClass distribution by port:")
print((pclass_emb * 100).round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
surv_emb["rate"].plot(kind="bar", ax=axes[0], color=SURVIVE_COLORS[1], rot=0)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].axhline(df["survived"].mean(), ls="--", color="gray")
axes[0].set_title("Survival Rate by Port")
axes[0].set_xlabel("Port of Embarkation (C=Cherbourg, Q=Queenstown, S=Southampton)")
pclass_emb.plot(kind="bar", ax=axes[1], stacked=True, rot=0,
                color=["#d62728","#ff7f0e","#2ca02c"])
axes[1].set_title("Class Mix by Port")
axes[1].set_xlabel("Port")
axes[1].legend(title="Pclass", labels=["1st","2nd","3rd"])
plt.suptitle("Observation 9: Embarkation Port", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "obs9_embarkation.png", dpi=110)
plt.show(); plt.close()

### Observation 10 — Social Status (Title) vs. Survival
> **"Passengers with titles 'Mr' (predominantly male, all classes) had only a 15.7%
> survival rate vs 70.3% for 'Miss' and 79.2% for 'Mrs', reinforcing that sex
> and social standing jointly determined evacuation priority."**

In [ ]:
TITLE_MAP = {
    "Mr":    "Mr",   "Miss": "Miss",  "Mrs":  "Mrs",
    "Master":"Master","Dr":  "Rare",   "Rev":  "Rare",
    "Col":   "Rare",  "Major":"Rare",  "Mlle": "Miss",
    "Mme":   "Mrs",   "Ms":   "Miss",  "Lady": "Rare",
    "Jonkheer":"Rare","Don":  "Rare",  "Sir":  "Rare",
    "Capt":  "Rare", "the Countess":"Rare",
}
df["title_group"] = df["title"].str.strip().map(TITLE_MAP).fillna("Rare")

surv_title = df.groupby("title_group")["survived"].agg(["mean","count"])
surv_title.columns = ["rate","count"]
surv_title = surv_title.sort_values("rate", ascending=False)
print(surv_title.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
colors = [SURVIVE_COLORS[1] if r >= 0.5 else SURVIVE_COLORS[0]
          for r in surv_title["rate"]]
bars = ax.barh(surv_title.index, surv_title["rate"],
               color=colors, edgecolor="white")
ax.axvline(df["survived"].mean(), ls="--", color="gray", label="Overall avg")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
for bar, (_, row) in zip(bars, surv_title.iterrows()):
    ax.text(row["rate"] + 0.005, bar.get_y() + bar.get_height()/2,
            f"{row['rate']:.1%}  n={row['count']}", va="center", fontsize=10)
ax.set_xlabel("Survival Rate")
ax.set_title("Observation 10: Survival Rate by Title Group", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "obs10_title_survival.png", dpi=110)
plt.show(); plt.close()

---
## §5 — Hypothesis Testing: Does Passenger Class Predict Survival?

**Research question:** Is there a statistically significant difference in survival rates
between 1st-class and 3rd-class passengers?

**Hypotheses:**
- **H₀:** P(survive | Pclass=1) = P(survive | Pclass=3)
- **H₁:** P(survive | Pclass=1) ≠ P(survive | Pclass=3)
- **α = 0.05** (two-tailed)

We apply three complementary tests to triangulate the finding:
1. Independent samples t-test (tests means of binary outcome — valid for large n by CLT)
2. Chi-square test of independence (most appropriate for count data)
3. Mann-Whitney U (non-parametric, distribution-free)


### 5.1 — Assumption Checks (Normality & Variance Equality)

In [ ]:
group1 = df[df["pclass"] == 1]["survived"].dropna()
group3 = df[df["pclass"] == 3]["survived"].dropna()

print("=" * 60)
print("GROUP SUMMARY")
print("=" * 60)
for name, g in [("Pclass 1", group1), ("Pclass 3", group3)]:
    print(f"{name}: n={len(g)}, mean={g.mean():.3f}, std={g.std():.3f}")

print("\n── Levene Test (variance equality) ──")
lev_stat, lev_p = stats.levene(group1, group3)
equal_var = lev_p > 0.05
print(f"Levene statistic : {lev_stat:.4f}")
print(f"p-value          : {lev_p:.4f}")
print(f"Equal variances? : {'Yes (p > 0.05)' if equal_var else 'No (p ≤ 0.05) → use Welch t-test'}")

print("\n── Shapiro-Wilk Normality Test ──")
print("Note: survival is binary (0/1) — normality fails by design.")
print("The CLT guarantees the sampling distribution of the mean is ~Normal for large n.")
for name, g in [("Pclass 1", group1), ("Pclass 3", group3)]:
    sw_stat, sw_p = stats.shapiro(g[:500] if len(g)>500 else g)
    print(f"{name}: W={sw_stat:.4f}, p={sw_p:.4e}  "
          f"({'✅ normal' if sw_p>0.05 else '❌ non-normal (expected for binary)'})")

print("\n→ Normality fails (expected for binary outcomes).")
print("  CLT applies: n1=%d, n3=%d — t-test is asymptotically valid." % (len(group1), len(group3)))
print("  Chi-square and Mann-Whitney U are reported as robust cross-checks.")

# QQ plots
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, g) in zip(axes, [("Pclass 1 Survival", group1), ("Pclass 3 Survival", group3)]):
    stats.probplot(g, dist="norm", plot=ax)
    ax.set_title(f"Q-Q Plot: {name}")
plt.suptitle("Assumption Check: Q-Q Plots (binary data → non-normal)", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "ttest_qq_plots.png", dpi=110)
plt.show(); plt.close()

### 5.2 — Independent Samples t-test  (Welch correction applied)

In [ ]:
print("=" * 60)
print("INDEPENDENT SAMPLES t-TEST  (equal_var=False → Welch)")
print("=" * 60)

t_stat, p_val = stats.ttest_ind(group1, group3, equal_var=False)
print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_val:.2e}")
print(f"Significant : {'Yes ✅  reject H₀' if p_val < 0.05 else 'No ❌  fail to reject H₀'}")

print("\n── Cohen's d (effect size) ──")
n1, n3    = len(group1), len(group3)
pool_std  = np.sqrt(((n1-1)*group1.std()**2 + (n3-1)*group3.std()**2) / (n1+n3-2))
cohens_d  = (group1.mean() - group3.mean()) / pool_std
print(f"Cohen's d : {cohens_d:.3f}")
magnitude = ("small (<0.2)" if abs(cohens_d)<0.2
             else "small-medium (0.2–0.5)" if abs(cohens_d)<0.5
             else "medium (0.5–0.8)" if abs(cohens_d)<0.8
             else "large (>0.8)")
print(f"Magnitude : {magnitude}")

print("\n── 95% Confidence Interval for Δ mean (Pclass 1 − Pclass 3) ──")
diff  = group1.mean() - group3.mean()
se    = np.sqrt(group1.var()/n1 + group3.var()/n3)
df_w  = (group1.var()/n1 + group3.var()/n3)**2 / (
         (group1.var()/n1)**2/(n1-1) + (group3.var()/n3)**2/(n3-1))
ci_lo, ci_hi = stats.t.interval(0.95, df=df_w, loc=diff, scale=se)
print(f"Δ mean        : {diff:+.3f}  (Pclass1 {group1.mean():.3f} vs Pclass3 {group3.mean():.3f})")
print(f"95% CI        : [{ci_lo:+.3f},  {ci_hi:+.3f}]")

### 5.3 — Chi-Square Test of Independence  (most appropriate for count data)

In [ ]:
print("=" * 60)
print("CHI-SQUARE TEST OF INDEPENDENCE")
print("=" * 60)

contingency = pd.crosstab(
    df[df["pclass"].isin([1, 3])]["pclass"],
    df[df["pclass"].isin([1, 3])]["survived"],
    rownames=["pclass"], colnames=["survived"]
)
print("Contingency table (Pclass 1 vs 3):")
print(contingency.to_string())

chi2, chi_p, dof, expected = stats.chi2_contingency(contingency)
print(f"\nchi2 statistic : {chi2:.4f}")
print(f"p-value        : {chi_p:.2e}")
print(f"Degrees of freedom : {dof}")
print(f"Significant    : {'Yes ✅' if chi_p < 0.05 else 'No ❌'}")

# Cramér's V (effect size for chi-square)
n   = contingency.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
print(f"\nCramér's V (effect size) : {cramers_v:.3f}")
print(f"  (<0.10 negligible | 0.10–0.30 small | 0.30–0.50 medium | >0.50 large)")

### 5.4 — Mann-Whitney U Test  (non-parametric, distribution-free)

In [ ]:
print("=" * 60)
print("MANN-WHITNEY U TEST  (non-parametric alternative)")
print("=" * 60)

u_stat, mw_p = stats.mannwhitneyu(group1, group3, alternative="two-sided")
print(f"U statistic : {u_stat:.1f}")
print(f"p-value     : {mw_p:.2e}")
print(f"Significant : {'Yes ✅  reject H₀' if mw_p < 0.05 else 'No ❌'}")

# Rank-biserial correlation r (effect size for Mann-Whitney)
r_rb = 1 - (2 * u_stat) / (n1 * n3)
print(f"\nRank-biserial r (effect size) : {r_rb:.3f}")

print("\n" + "=" * 60)
print("COMBINED TEST SUMMARY")
print("=" * 60)
results = [
    ("t-test (Welch)",       p_val,  cohens_d,  "Cohen's d"),
    ("Chi-square",           chi_p,  cramers_v, "Cramér's V"),
    ("Mann-Whitney U",       mw_p,   r_rb,      "Rank-biserial r"),
]
print(f"{'Test':<22} {'p-value':>12}  {'Effect':>8}  {'Metric':<15}  {'Sig?'}")
print("-" * 70)
for name, p, eff, metric in results:
    sig = "✅ Yes" if p < 0.05 else "❌ No"
    print(f"{name:<22} {p:>12.2e}  {eff:>8.3f}  {metric:<15}  {sig}")

### 5.5 — Interpretation (paste into report)

In [ ]:
print("""
INTERPRETATION TEMPLATE
────────────────────────────────────────────────────────────────────────
We tested whether passenger class (1st vs 3rd) was significantly associated
with survival on the RMS Titanic.

FINDING: All three statistical tests converge on strong evidence against H₀.

  • Welch t-test: t = {t:.2f}, p < 0.001
    The mean survival rate for 1st-class passengers ({m1:.1%}) was
    {diff_pct:.1f} percentage points higher than for 3rd-class ({m3:.1%}).
    The 95% CI for this difference is [{ci_lo:.1f}%, {ci_hi:.1f}%], which does
    not include 0 — consistent with a real effect.

  • Cohen's d = {d:.2f} (large effect).
    A value above 0.8 indicates a practically, not just statistically,
    significant difference.

  • Chi-square: χ²({dof}) = {chi2_val:.1f}, p < 0.001, Cramér's V = {cv:.2f}
    The association between class and survival is strong.

  • Mann-Whitney U: p < 0.001, rank-biserial r = {r:.2f} (large effect)

CONCLUSION: We reject H₀. Passenger class is a highly significant predictor
of survival (p < 0.001, large effect size by all metrics).  Class-based
inequality in lifeboat access is not attributable to chance.

LIMITATION: This is an observational dataset; confounders (sex, age, fare,
embarkation) partially mediate the class effect.  A multivariate logistic
regression controls for these co-variates.
────────────────────────────────────────────────────────────────────────
""".format(
    t=t_stat, m1=group1.mean(), m3=group3.mean(),
    diff_pct=(group1.mean()-group3.mean())*100,
    ci_lo=ci_lo*100, ci_hi=ci_hi*100,
    d=cohens_d, chi2_val=chi2, dof=dof, cv=cramers_v, r=r_rb,
))

---
## §6 — Executive Summary, Export & GitHub Pages Publish

### 6.1 — Executive Summary

**Titanic EDA — Key Findings (fill in your computed values)**

This analysis of 891 Titanic passengers reveals that survival was far from random.
Overall, only **38.4%** survived, but outcomes varied dramatically by demographic:

1. **Sex was the strongest predictor.** Females survived at 74.2% vs males at 18.9% (3.9× ratio).
2. **Class created a hierarchy of survival.** 1st class: 63.0%; 2nd class: 47.3%; 3rd class: 24.2%.
3. **1st-class females had near-certain survival (96.5%)** — effectively guaranteed by both status and gender.
4. **Children under 10 benefited from evacuation priority** (~58% survival).
5. **Traveling alone was risky.** Solo passengers survived at 30.4% vs 50.5% for small family groups.
6. **Cherbourg embarkees (55.4% survival)** reflect their higher proportion of 1st-class passengers.
7. **Fare correlates with survival**, but this is largely a proxy for passenger class.
8. **19.9% of ages are missing** — imputation strategy affects age-based conclusions.
9. **The "Mr" title group (517 passengers, mainly adult males) had only 15.7% survival.**
10. **Statistical test confirms class effect** (p < 0.001, Cohen's d = large) — not due to chance.

**Recommended next steps:** Fit a logistic regression with all features; build a Random Forest
to rank feature importances; compare survival prediction with a held-out test set.

In [ ]:
# ── Save cleaned sample for GitHub Pages ─────────────────────────────────────
cols_keep = ["passenger_id","survived","pclass","sex","age","fare",
             "family_size","is_alone","embarked","title_group"]
sample = df[cols_keep].head(100)
sample.to_csv(DATA_DIR / "titanic_clean_sample.csv", index=False)
print(f"✅ Sample CSV saved: data/titanic_clean_sample.csv  ({len(sample)} rows)")

# ── Export figures list ────────────────────────────────────────────────────────
figs = sorted(FIG_DIR.glob("*.png"))
print(f"\n✅ Figures saved ({len(figs)}):")
for f in figs:
    print(f"   {f.name}  ({f.stat().st_size//1024} KB)")

In [ ]:
# ── Build docs/index.html for GitHub Pages ───────────────────────────────────
INDEX_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Titanic EDA — Day 4</title>
  <style>
    body {{ font-family: system-ui, sans-serif; max-width: 900px; margin: 40px auto;
            padding: 0 20px; color: #333; line-height: 1.6; }}
    h1   {{ color: #1a1a2e; border-bottom: 3px solid #4c72b0; padding-bottom: 8px; }}
    h2   {{ color: #4c72b0; margin-top: 36px; }}
    img  {{ max-width: 100%; border: 1px solid #ddd; border-radius: 6px;
            margin: 12px 0; box-shadow: 0 2px 6px rgba(0,0,0,.1); }}
    .obs {{ background: #f4f8ff; border-left: 4px solid #4c72b0;
            padding: 12px 16px; border-radius: 0 6px 6px 0; margin: 16px 0; }}
    .stat{{ background: #fff8e1; border-left: 4px solid #ffa000;
            padding: 12px 16px; border-radius: 0 6px 6px 0; margin: 16px 0; }}
    a    {{ color: #4c72b0; }}
    footer {{ margin-top: 60px; font-size: 0.85em; color: #888; }}
  </style>
</head>
<body>
  <h1>🚢 Titanic EDA Report — Day 4</h1>
  <p>Statistical analysis of the Titanic passenger dataset.
     <a href="titanic_profile.html">Full ydata-profiling report →</a></p>

  <h2>Top 10 Observations</h2>

  <div class="obs"><strong>1.</strong> Only <strong>38.4%</strong> of 891 passengers survived.</div>
  <img src="../figures/obs1_overall_survival.png" alt="Overall survival">

  <div class="obs"><strong>2.</strong> Females survived at 3.9× the rate of males (74.2% vs 18.9%).</div>
  <img src="../figures/obs2_survival_sex.png" alt="Survival by sex">

  <div class="obs"><strong>3.</strong> 1st-class passengers survived at 2.6× the rate of 3rd-class (63% vs 24%).</div>
  <img src="../figures/obs3_survival_class.png" alt="Survival by class">

  <div class="obs"><strong>4.</strong> Median age was 28; children under 10 had notably higher survival rates.</div>
  <img src="../figures/obs4_age_distribution.png" alt="Age distribution">

  <div class="obs"><strong>5.</strong> Fares are heavily right-skewed; median £14.5, max £512.</div>
  <img src="../figures/obs5_fare_distribution.png" alt="Fare distribution">

  <div class="obs"><strong>6.</strong> 1st-class females had 96.5% survival; 3rd-class males had 13.5%.</div>
  <img src="../figures/obs6_sex_class_heatmap.png" alt="Class × sex heatmap">

  <div class="obs"><strong>7.</strong> Survivors had a younger median age (28 vs 30).</div>
  <img src="../figures/obs7_age_violin.png" alt="Age violin plot">

  <div class="obs"><strong>8.</strong> Small family groups (2–4) had the highest survival (50.5%).</div>
  <img src="../figures/obs8_family_size.png" alt="Family size survival">

  <div class="obs"><strong>9.</strong> Cherbourg embarkees had the highest survival (55.4%).</div>
  <img src="../figures/obs9_embarkation.png" alt="Embarkation port">

  <div class="obs"><strong>10.</strong> 'Mr' title group: 15.7% survival vs 79.2% for 'Mrs'.</div>
  <img src="../figures/obs10_title_survival.png" alt="Title survival">

  <h2>Statistical Test: Class 1 vs Class 3 Survival</h2>
  <div class="stat">
    <strong>Welch t-test:</strong> p &lt; 0.001 | Cohen's d = large<br>
    <strong>Chi-square:</strong> p &lt; 0.001 | Cramér's V = strong<br>
    <strong>Mann-Whitney U:</strong> p &lt; 0.001 | rank-biserial r = large<br>
    <em>Conclusion: Passenger class is a highly significant predictor of survival (not due to chance).</em>
  </div>

  <h2>Links</h2>
  <ul>
    <li><a href="titanic_profile.html">Full ydata-profiling HTML report</a></li>
    <li><a href="https://github.com/YOUR_USERNAME/ai-ds-roadmap">GitHub Repository</a></li>
  </ul>

  <footer>Generated as part of the AI & DS 21-Day Roadmap — Day 4</footer>
</body>
</html>"""

with open(DOCS_DIR / "index.html", "w") as f:
    f.write(INDEX_HTML)

print(f"✅ docs/index.html written ({len(INDEX_HTML)//1024} KB)")
print(f"   Preview: open docs/index.html in your browser")

In [ ]:
# ── GitHub Pages publish checklist ────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════╗
║          GITHUB PAGES PUBLISH CHECKLIST                      ║
╠══════════════════════════════════════════════════════════════╣
║  AUTOMATED (run in terminal from repo root):                  ║
║                                                               ║
║  # 1. Export this notebook to HTML                            ║
║  jupyter nbconvert --to html day4_eda/titanic_eda.ipynb       ║
║  cp day4_eda/titanic_eda.html day4_eda/docs/                  ║
║                                                               ║
║  # 2. Verify docs/ folder contents                            ║
║  ls day4_eda/docs/                                            ║
║  #  index.html  titanic_profile.html  titanic_eda.html        ║
║                                                               ║
║  # 3. Commit and push                                         ║
║  git add day4_eda/                                            ║
║  git commit -m "feat: Day 4 — Titanic EDA + stats ✅"         ║
║  git push origin main                                         ║
║                                                               ║
║  MANUAL (once, in GitHub UI):                                 ║
║  4. Go to repo → Settings → Pages                            ║
║  5. Source: Deploy from branch                               ║
║  6. Branch: main  /  Folder: /day4_eda/docs                  ║
║  7. Click Save                                               ║
║  8. Wait 1-3 min → URL appears:                              ║
║     https://YOUR_USERNAME.github.io/ai-ds-roadmap/           ║
║                                                               ║
║  DELIVERABLE CHECKLIST                                        ║
║  ☐ titanic_eda.ipynb runs end-to-end (0 errors)              ║
║  ☐ titanic_profile.html generated (data/ folder)             ║
║  ☐ data/titanic_clean_sample.csv exported                    ║
║  ☐ 10 observations written with supporting plots             ║
║  ☐ t-test + chi-square + Mann-Whitney all run                ║
║  ☐ docs/index.html live at GitHub Pages URL                  ║
╚══════════════════════════════════════════════════════════════╝
""")